# Part2 RegGS: All-Non-Train Test Rendering and Evaluation

这个 notebook 用于**单独执行 test 渲染+评测**，支持：

1. 使用 `all_non_train` 协议（测试集 = 所有非训练帧）
2. 按与最近训练帧距离筛选要渲染的 test 帧
3. 保存渲染图、可选保存 GT 图、可选保存 test pose
4. 输出独立的 metrics json，避免覆盖已有结果

注意：这是一个独立 notebook，不会修改 RegGS 库代码。

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import torch
import torchvision
from torchvision.utils import save_image

# -------- Workspace Paths (standalone notebook) --------
CV_ROOT = Path('/home/bzhang512/CV_Project')
PART2_ROOT = CV_ROOT / 'part2'
REGGS_ROOT = CV_ROOT / 'third_party' / 'RegGS'
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2'
CONFIG_ROOT = PART2_ROOT / 'configs'

assert REGGS_ROOT.exists(), f'RegGS root not found: {REGGS_ROOT}'
assert OUTPUT_ROOT.exists(), f'Output root not found: {OUTPUT_ROOT}'
assert CONFIG_ROOT.exists(), f'Config root not found: {CONFIG_ROOT}'

print('CV_ROOT =', CV_ROOT)
print('REGGS_ROOT =', REGGS_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('CONFIG_ROOT =', CONFIG_ROOT)

## 1) 选择运行目录和配置

- `DATASET_KEY`: 对应 `output/part2/<dataset_key>`
- `RUN_NAME`: 具体 run 目录名
- `CONFIG_PATH`: 默认使用 `part2/configs/<RUN_NAME>.yaml`，不存在时回退到 `RUN_OUTPUT/config.yaml`

In [ ]:
# -------- Run Selection --------
# 可选: 're10k_1', 'dl3dv_2', '405841'
DATASET_KEY = '405841'

# 你可以直接改成目标 run
RUN_NAME = 'reggs_405841_scene_C_dl3dv-ckpt_sr30_nv12_sm2_stable_v1'

# 也可留空并自动选择该数据集目录下按修改时间最新的 run
AUTO_PICK_LATEST_IF_EMPTY = False

dataset_output_dir = OUTPUT_ROOT / DATASET_KEY
assert dataset_output_dir.exists(), f'Dataset output dir missing: {dataset_output_dir}'

if AUTO_PICK_LATEST_IF_EMPTY or (RUN_NAME is None) or (str(RUN_NAME).strip() == ''):
    candidates = [p for p in dataset_output_dir.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No run dirs found under: {dataset_output_dir}')
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    RUN_NAME = candidates[0].name

RUN_OUTPUT = dataset_output_dir / RUN_NAME
assert RUN_OUTPUT.exists(), f'Run output not found: {RUN_OUTPUT}'

CONFIG_PATH = CONFIG_ROOT / f'{RUN_NAME}.yaml'
if not CONFIG_PATH.exists():
    fallback_cfg = RUN_OUTPUT / 'config.yaml'
    if fallback_cfg.exists():
        CONFIG_PATH = fallback_cfg
    else:
        raise FileNotFoundError(
            f'Config not found in configs and run dir: {CONFIG_ROOT / (RUN_NAME + ".yaml")}, {fallback_cfg}'
        )

print('DATASET_KEY =', DATASET_KEY)
print('RUN_NAME =', RUN_NAME)
print('RUN_OUTPUT =', RUN_OUTPUT)
print('CONFIG_PATH =', CONFIG_PATH)

## 2) Test 协议与筛帧参数

你主要改这个单元里的参数：

- `TEST_PROTOCOL`: 通常设为 `all_non_train`
- `OUTPUT_TAG`: 控制输出目录和 json 文件名后缀
- `SELECTION_MODE`: `all` / `gap_step` / `manual`
- `MIN_GAP_TO_TRAIN` 等：用于 `gap_step` 筛选
- `DO_POSE_OPT`: 开启更准但更慢；关闭更快
- `SAVE_TEST_POSES`: 是否额外保存 test pose 到 ckpt

In [ ]:
# -------- Test Protocol and Frame Selection --------
TEST_PROTOCOL = 'all_non_train'   # 'sampled_test' or 'all_non_train'
OUTPUT_TAG = 'all_non_train_subset_v1'

# 选择模式:
# - 'all': 使用协议返回的全部 test 帧
# - 'gap_step': 按与最近 train 帧距离筛选
# - 'manual': 仅使用 MANUAL_FRAME_IDS
SELECTION_MODE = 'gap_step'

# gap_step 参数
MIN_GAP_TO_TRAIN = 20
MAX_GAP_TO_TRAIN = None   # 例如 80；None 表示不设上限
TAKE_EVERY_N = 3          # 在候选中每 N 帧保留 1 帧
MAX_KEEP = None           # 例如 50；None 表示不截断

# manual 参数
MANUAL_FRAME_IDS = []     # 例如 [15, 45, 75]

# 额外包含/排除
FORCE_INCLUDE = []
FORCE_EXCLUDE = []

# 渲染和保存控制
DO_POSE_OPT = True
SAVE_GT_IMAGES = True
SAVE_TEST_POSES = False

print('TEST_PROTOCOL =', TEST_PROTOCOL)
print('OUTPUT_TAG =', OUTPUT_TAG)
print('SELECTION_MODE =', SELECTION_MODE)
print('DO_POSE_OPT =', DO_POSE_OPT)
print('SAVE_GT_IMAGES =', SAVE_GT_IMAGES)
print('SAVE_TEST_POSES =', SAVE_TEST_POSES)

In [ ]:
# -------- Import RegGS Modules and Build Evaluator --------
if str(REGGS_ROOT) not in sys.path:
    sys.path.insert(0, str(REGGS_ROOT))

from src.evaluation.evaluator import Evaluator
from src.utils.utils import get_render_settings, render_gaussian_model
from src.utils.metrics_utils import compute_psnr, compute_ssim, compute_lpips

if TEST_PROTOCOL not in {'sampled_test', 'all_non_train'}:
    raise ValueError('TEST_PROTOCOL must be sampled_test/all_non_train')

evaluator = Evaluator(
    Path(RUN_OUTPUT),
    Path(CONFIG_PATH),
    test_protocol=TEST_PROTOCOL,
    test_output_tag=OUTPUT_TAG,
)

if evaluator.refined_gaussians is None:
    raise RuntimeError('No gaussian model found in run output.')

print('Evaluator ready.')
print('train frames =', len(evaluator.train_frame_ids))
print('test frames (protocol) =', len(evaluator.test_frame_ids))

In [ ]:
# -------- Build Selected Test Frame List --------
train_ids = np.array(evaluator.train_frame_ids, dtype=int)
test_ids = np.array(evaluator.test_frame_ids, dtype=int)

if test_ids.size == 0:
    raise RuntimeError('No test frames found under current protocol.')

# 每个 test 帧到最近 train 帧的距离
dist_to_nearest_train = np.min(np.abs(test_ids[:, None] - train_ids[None, :]), axis=1)
dist_map = {int(fid): int(d) for fid, d in zip(test_ids.tolist(), dist_to_nearest_train.tolist())}

if SELECTION_MODE == 'all':
    selected_ids = test_ids.copy()
elif SELECTION_MODE == 'manual':
    selected_ids = np.array(sorted(set(int(x) for x in MANUAL_FRAME_IDS)), dtype=int)
elif SELECTION_MODE == 'gap_step':
    mask = dist_to_nearest_train >= int(MIN_GAP_TO_TRAIN)
    if MAX_GAP_TO_TRAIN is not None:
        mask = mask & (dist_to_nearest_train <= int(MAX_GAP_TO_TRAIN))
    selected_ids = test_ids[mask]
    if TAKE_EVERY_N is not None and int(TAKE_EVERY_N) > 1:
        selected_ids = selected_ids[::int(TAKE_EVERY_N)]
    if MAX_KEEP is not None:
        selected_ids = selected_ids[:int(MAX_KEEP)]
else:
    raise ValueError('SELECTION_MODE must be one of all/manual/gap_step')

if FORCE_INCLUDE:
    selected_ids = np.unique(np.concatenate([selected_ids, np.array([int(x) for x in FORCE_INCLUDE])]))
if FORCE_EXCLUDE:
    excluded = set(int(x) for x in FORCE_EXCLUDE)
    selected_ids = np.array([x for x in selected_ids if int(x) not in excluded], dtype=int)

selected_ids = np.array(sorted(int(x) for x in selected_ids.tolist()), dtype=int)
if selected_ids.size == 0:
    raise RuntimeError('No selected test frames. Please relax selection parameters.')

selected_dist = [dist_map.get(int(fid), int(np.min(np.abs(train_ids - int(fid))))) for fid in selected_ids]

print('train count =', len(train_ids))
print('test count (protocol) =', len(test_ids))
print('selected count =', len(selected_ids))
print('selected head =', selected_ids[:20].tolist())
print('selected gap head =', selected_dist[:20])

In [ ]:
# -------- Render Selected Test Frames and Evaluate --------
run_dir = Path(RUN_OUTPUT)
render_dir = run_dir / evaluator._test_render_dir()
gt_dir = run_dir / evaluator._test_gt_dir()
metrics_path = run_dir / evaluator._test_metrics_filename()
pose_path = run_dir / f'estimated_test_c2w_{evaluator.test_output_label}.ckpt'

render_dir.mkdir(parents=True, exist_ok=True)
if SAVE_GT_IMAGES:
    gt_dir.mkdir(parents=True, exist_ok=True)

print('render_dir =', render_dir)
if SAVE_GT_IMAGES:
    print('gt_dir =', gt_dir)
print('metrics_path =', metrics_path)
if SAVE_TEST_POSES:
    print('pose_path =', pose_path)

device = evaluator.device
dataset = evaluator.dataset
model = evaluator.refined_gaussians
transform = torchvision.transforms.ToTensor()

gt_c2ws = torch.tensor(dataset.poses).to(device)
est_train_c2ws = evaluator.estimated_c2ws
est_test_c2ws = evaluator.compute_estimated_test_c2ws(
    gt_c2ws, est_train_c2ws, train_ids, selected_ids
)

metrics_rows = []
saved_poses = []

for idx, frame_id in enumerate(selected_ids):
    frame_id = int(frame_id)
    est_c2w = est_test_c2ws[idx]

    if DO_POSE_OPT:
        est_c2w = evaluator.optimize_estimated_c2w(model, est_c2w, frame_id)

    with torch.no_grad():
        w2c = torch.inverse(est_c2w)
        render_settings = get_render_settings(
            dataset.width, dataset.height, dataset.intrinsics, w2c.detach().cpu().numpy()
        )
        render_dict = render_gaussian_model(model, render_settings)

        gt_color = transform(dataset[frame_id][1]).to(device)
        pred_color = render_dict['color'].clamp(0, 1)

        psnr = compute_psnr(gt_color, pred_color).item()
        ssim = compute_ssim(gt_color, pred_color).item()
        lpips = compute_lpips(gt_color, pred_color).item()

    save_image(pred_color, render_dir / f'color_{frame_id:04d}.png')
    if SAVE_GT_IMAGES:
        save_image(gt_color, gt_dir / f'gt_{frame_id:04d}.png')

    if SAVE_TEST_POSES:
        saved_poses.append(est_c2w.detach().cpu())

    gap = int(np.min(np.abs(train_ids - frame_id)))
    metrics_rows.append({
        'frame_id': f'{frame_id:04d}',
        'frame_id_int': frame_id,
        'dist_to_nearest_train': gap,
        'psnr': psnr,
        'ssim': ssim,
        'lpips': lpips,
    })

    print(f'[{idx + 1:04d}/{len(selected_ids):04d}] frame={frame_id:04d} gap={gap:03d} '
          f'PSNR={psnr:.2f} SSIM={ssim:.3f} LPIPS={lpips:.3f}')

avg_psnr = float(np.mean([r['psnr'] for r in metrics_rows]))
avg_ssim = float(np.mean([r['ssim'] for r in metrics_rows]))
avg_lpips = float(np.mean([r['lpips'] for r in metrics_rows]))

payload = {
    'test_protocol': TEST_PROTOCOL,
    'output_tag': evaluator.test_output_label,
    'selection_mode': SELECTION_MODE,
    'selection_params': {
        'min_gap_to_train': MIN_GAP_TO_TRAIN,
        'max_gap_to_train': MAX_GAP_TO_TRAIN,
        'take_every_n': TAKE_EVERY_N,
        'max_keep': MAX_KEEP,
        'manual_frame_ids': MANUAL_FRAME_IDS,
        'force_include': FORCE_INCLUDE,
        'force_exclude': FORCE_EXCLUDE,
        'do_pose_opt': DO_POSE_OPT,
        'save_gt_images': SAVE_GT_IMAGES,
        'save_test_poses': SAVE_TEST_POSES,
    },
    'n_train_frames': int(len(train_ids)),
    'n_test_frames_protocol': int(len(test_ids)),
    'n_test_frames_selected': int(len(selected_ids)),
    'avg_psnr': avg_psnr,
    'avg_ssim': avg_ssim,
    'avg_lpips': avg_lpips,
    'metrics': metrics_rows,
}

with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

if SAVE_TEST_POSES:
    torch.save(
        {
            'frame_ids': [int(x) for x in selected_ids.tolist()],
            'estimated_test_c2w': torch.stack(saved_poses, dim=0) if len(saved_poses) > 0 else torch.empty(0, 4, 4),
            'test_protocol': TEST_PROTOCOL,
            'output_tag': evaluator.test_output_label,
        },
        pose_path
    )

print('Done.')
print('avg_psnr =', avg_psnr)
print('avg_ssim =', avg_ssim)
print('avg_lpips =', avg_lpips)

## 3) 使用说明（简版）

1. 先改 `DATASET_KEY` 和 `RUN_NAME`，确认 `RUN_OUTPUT` 与 `CONFIG_PATH`
2. 将 `TEST_PROTOCOL` 设为 `all_non_train`
3. 调整 `SELECTION_MODE` 与筛帧参数
4. 执行最后一个渲染单元
5. 结果在：
   - 渲染图：`RUN_OUTPUT/test_<output_tag>/`
   - GT 图：`RUN_OUTPUT/gt_<output_tag>/`（若开启）
   - 指标：`RUN_OUTPUT/eval_test_<output_tag>.json`
   - pose：`RUN_OUTPUT/estimated_test_c2w_<output_tag>.ckpt`（若开启）